# Stochastic Interpolants: Deterministic ODE vs. Stochastic SDE Debiasing

This self-contained CPU notebook demonstrates unpaired debiasing of a 1-D
bimodal periodic physical field using `swirl_dynamics.lib.flow_matching` and
`swirl_dynamics.projects.debiasing.stochastic_interpolants.trainers`:

* **Interpolant:** `LinearInterpolantSinusoidalNoise(sigma)` with smooth bridge
  $x_t = (1 - t) x_0 + t x_1 + \sigma \sin^2(\pi t)\thinspace z$.
* **Deterministic ODE:** Trains `StochasticInterpolantModel` with
  `velocity_loss` and integrates $\frac{dx_t}{dt} = v_\theta(x_t, t)$.
* **Stochastic Flow+Score SDE:** Trains `StochasticInterpolantFlowScoreModel`
  with `velocity_loss` + `denoising_loss` ($\eta_\theta \approx \mathbb{E}[z \mid x_t]$)
  and integrates the singularity-free Euler-Maruyama SDE
  $$
  d x_t = \big(v_\theta(x_t, t) - c\thinspace \eta_\theta(x_t, t)\big)\thinspace dt + \sqrt{2 c\thinspace \gamma(t)}\thinspace dW_t.
  $$
* **Diagnostics:** Compares 1-D Wasserstein-1 distance ($W_1$) and mean
  log-power-spectrum error between raw, ODE-debiased, and SDE-debiased ensembles.


## 1. Installation & Imports


In [ ]:
!pip install -q "flax>=0.12.8" git+https://github.com/google-research/swirl-dynamics.git

In [ ]:
import os
import tempfile

import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import ml_collections
import numpy as np
import optax
from scipy import stats
from swirl_dynamics.lib import flow_map
from swirl_dynamics.lib import flow_matching
from swirl_dynamics.projects.debiasing.stochastic_interpolants import trainers

## 2. Configuration


In [ ]:
config = ml_collections.ConfigDict()
config.seed = 42
config.grid_size = 32
config.num_train_samples = 384
config.num_eval_samples = 64
config.batch_size = 32
config.sigma = 0.25
config.hidden_channels = 24
config.learning_rate = 3e-3
config.ema_decay = 0.98
config.num_train_steps = 80
config.num_integration_steps = 20
config.sde_diffusion_scale = 0.2
workdir = tempfile.mkdtemp(prefix="si_simple_demo_")

## 3. Synthetic Unpaired Bimodal Periodic Field


In [ ]:
def make_unpaired_fields(num_samples, grid_size, seed):
  rng = np.random.default_rng(seed)
  x_grid = np.linspace(0.0, 2.0 * np.pi, grid_size, endpoint=False)
  # Biased low-order source (damped higher harmonic + shifted mean).
  phase_a = rng.uniform(0.0, 2.0 * np.pi, size=(num_samples, 1))
  mode_a = rng.choice([-0.8, 0.8], size=(num_samples, 1))
  x_0 = mode_a + 0.7 * np.sin(x_grid[None, :] + phase_a)
  x_0 += 0.1 * np.sin(2.0 * x_grid[None, :] + 2.0 * phase_a)
  # Unpaired high-fidelity reference (stronger bimodal split + 2nd harmonic).
  phase_b = rng.uniform(0.0, 2.0 * np.pi, size=(num_samples, 1))
  mode_b = rng.choice([-1.4, 1.4], size=(num_samples, 1))
  x_1 = mode_b + 1.0 * np.sin(x_grid[None, :] + phase_b)
  x_1 += 0.45 * np.sin(2.0 * x_grid[None, :] + 2.0 * phase_b)
  return {
      "x_0": x_0[..., None].astype(np.float32),
      "x_1": x_1[..., None].astype(np.float32),
  }


train_data = make_unpaired_fields(
    config.num_train_samples, config.grid_size, seed=config.seed
)
eval_data = make_unpaired_fields(
    config.num_eval_samples, config.grid_size, seed=config.seed + 1
)


def batch_iterator(dataset, batch_size, seed):
  rng = np.random.default_rng(seed)
  n = dataset["x_0"].shape[0]
  while True:
    idx_0 = rng.integers(0, n, size=(batch_size,))
    idx_1 = rng.integers(0, n, size=(batch_size,))
    yield {"x_0": dataset["x_0"][idx_0], "x_1": dataset["x_1"][idx_1]}

## 4. Compact Periodic Conv Backbones (Velocity & Joint Flow+Denoiser)


In [ ]:
class PeriodicResNet1D(nn.Module):
  hidden_channels: int = 24
  out_channels: int = 1
  dual_head: bool = False

  @nn.compact
  def __call__(self, x, sigma, *, is_training=False):
    del is_training
    t_emb = flow_map.FourierEmbedding(dims=self.hidden_channels)(sigma)
    t_bias = nn.Dense(self.hidden_channels)(t_emb)[:, None, :]
    h = nn.Conv(self.hidden_channels, (3,), padding="CIRCULAR")(x) + t_bias
    for _ in range(2):
      r = nn.silu(h)
      r = nn.Conv(self.hidden_channels, (3,), padding="CIRCULAR")(r) + t_bias
      r = nn.silu(r)
      r = nn.Conv(self.hidden_channels, (3,), padding="CIRCULAR")(r)
      h = h + r
    h = nn.silu(h)
    v = nn.Conv(self.out_channels, (3,), padding="CIRCULAR", name="v_head")(h)
    if not self.dual_head:
      return v
    eta = nn.Conv(self.out_channels, (3,), padding="CIRCULAR", name="s_head")(h)
    return v, eta

## 5. Train Deterministic ODE and Joint Flow+Score Models


In [ ]:
interp = flow_matching.LinearInterpolantSinusoidalNoise(sigma=config.sigma)
input_shape = (config.grid_size, 1)

ode_net = PeriodicResNet1D(
    hidden_channels=config.hidden_channels, out_channels=1, dual_head=False
)
ode_model = flow_matching.StochasticInterpolantModel(
    input_shape=input_shape,
    flow_model=ode_net,
    interpolant=interp,
    loss_stochastic_interpolant=flow_matching.velocity_loss,
)
ode_trainer = trainers.StochasticInterpolantTrainer(
    ema_decay=config.ema_decay,
    model=ode_model,
    rng=jax.random.PRNGKey(config.seed),
    optimizer=optax.adam(config.learning_rate),
)
ode_train = ode_trainer.train(
    batch_iterator(train_data, config.batch_size, seed=config.seed + 2),
    num_steps=config.num_train_steps,
).compute()

sde_net = PeriodicResNet1D(
    hidden_channels=config.hidden_channels, out_channels=1, dual_head=True
)
sde_model = flow_matching.StochasticInterpolantFlowScoreModel(
    input_shape=input_shape,
    flow_score_model=sde_net,
    interpolant=interp,
    loss_stochastic_interpolant_flow=flow_matching.velocity_loss,
    loss_stochastic_interpolant_score=flow_matching.denoising_loss,
)
sde_trainer = trainers.StochasticInterpolantFlowScoreTrainer(
    ema_decay=config.ema_decay,
    model=sde_model,
    rng=jax.random.PRNGKey(config.seed + 10),
    optimizer=optax.adam(config.learning_rate),
)
sde_train = sde_trainer.train(
    batch_iterator(train_data, config.batch_size, seed=config.seed + 3),
    num_steps=config.num_train_steps,
).compute()
print(
    f"ODE loss: {float(ode_train['train_loss']):.4f} | "
    f"SDE flow+denoiser loss: {float(sde_train['train_loss']):.4f}"
)

## 6. Integrate ODE & SDE and Compare $W_1$ & Power Spectra


In [ ]:
ode_fn = flow_matching.StochasticInterpolantModel.inference_fn(
    ode_trainer.train_state.ema_variables, ode_net
)
sde_fn = flow_matching.StochasticInterpolantFlowScoreModel.inference_fn(
    sde_trainer.train_state.ema_variables, sde_net
)


def integrate_ode(x_init, num_steps):
  dt = 1.0 / num_steps
  x = x_init
  for k in range(num_steps):
    t = jnp.full((x.shape[0],), (k + 0.5) * dt, dtype=jnp.float32)
    x = x + dt * ode_fn(x, t)
  return x


def integrate_sde(x_init, num_steps, diff_scale, rng):
  dt = 1.0 / num_steps
  x = x_init
  for k in range(num_steps):
    rng, step_rng = jax.random.split(rng)
    t_val = (k + 0.5) * dt
    t = jnp.full((x.shape[0],), t_val, dtype=jnp.float32)
    v, eta = sde_fn(x, t)
    gamma_t = float(interp.gamma(jnp.asarray(t_val, dtype=jnp.float32)))
    drift = v - diff_scale * eta
    noise = jax.random.normal(step_rng, x.shape)
    x = x + dt * drift + np.sqrt(2.0 * diff_scale * gamma_t * dt) * noise
  return x


x_eval_0 = jnp.asarray(eval_data["x_0"])
x_eval_1 = np.asarray(eval_data["x_1"])
samples_ode = np.asarray(integrate_ode(x_eval_0, config.num_integration_steps))
samples_sde = np.asarray(
    integrate_sde(
        x_eval_0,
        config.num_integration_steps,
        config.sde_diffusion_scale,
        jax.random.PRNGKey(config.seed + 99),
    )
)


def mean_spectrum(fields):
  fft = np.fft.rfft(fields[..., 0], axis=-1)
  return np.mean(np.abs(fft) ** 2, axis=0) + 1e-8


spec_ref = mean_spectrum(x_eval_1)
spec_raw = mean_spectrum(eval_data["x_0"])
spec_ode = mean_spectrum(samples_ode)
spec_sde = mean_spectrum(samples_sde)

metrics = {
    "w1_raw": float(
        stats.wasserstein_distance(eval_data["x_0"].ravel(), x_eval_1.ravel())
    ),
    "w1_ode": float(
        stats.wasserstein_distance(samples_ode.ravel(), x_eval_1.ravel())
    ),
    "w1_sde": float(
        stats.wasserstein_distance(samples_sde.ravel(), x_eval_1.ravel())
    ),
    "spec_err_raw": float(np.mean(np.abs(np.log10(spec_raw / spec_ref)))),
    "spec_err_ode": float(np.mean(np.abs(np.log10(spec_ode / spec_ref)))),
    "spec_err_sde": float(np.mean(np.abs(np.log10(spec_sde / spec_ref)))),
}
for name, val in metrics.items():
  print(f"{name:14s}: {val:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(
    eval_data["x_0"].ravel(), bins=30, density=True, alpha=0.4, label="Source"
)
axes[0].hist(x_eval_1.ravel(), bins=30, density=True, alpha=0.4, label="Target")
axes[0].hist(
    samples_ode.ravel(),
    bins=30,
    density=True,
    histtype="step",
    lw=2,
    label="ODE",
)
axes[0].hist(
    samples_sde.ravel(),
    bins=30,
    density=True,
    histtype="step",
    lw=2,
    label="SDE",
)
axes[0].set_title("Marginal PDF")
axes[0].legend(fontsize=8)

axes[1].semilogy(spec_ref, label="Target", lw=2)
axes[1].semilogy(spec_raw, "--", label="Source")
axes[1].semilogy(spec_ode, label="ODE")
axes[1].semilogy(spec_sde, label="SDE")
axes[1].set_title("Mean Energy Spectrum")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(workdir, "si_ode_vs_sde.png"), dpi=100)
plt.show()